# Cross-checking EMRI signals with `gw_response` against other tools

This notebook validates `gw_response`'s LISA TDI response against some lisa-simulation 
tools (i.e., [`lisagwresponse`](https://gitlab.in2p3.fr/lisa-simulation/gw-response) + 
[`pytdi`](https://pypi.org/project/pytdi/)) and ([`FastEMRIWaveforms`](https://github.com/BlackHolePerturbationToolkit/FastEMRIWaveforms)).

1. **Output**: do the two codes agree on the actual physics?
2. **Performance**: how does `gw_response`'s JAX/JIT-compiled approach
   compare to `lisagwresponse`'s time-domain, spline-interpolation approach?

**Requirement**: `lisagwresponse`, `pytdi` and `FastEMRIWaveforms` are *not* dependencies of `gw_response` (not even an optional one) — they are only needed to run this comparison notebook.
Install it yourself first:

```
pip install lisagwresponse
pip install pytdi
pip install git+https://github.com/BlackHolePerturbationToolkit/FastEMRIWaveforms
```


In [1]:
import time

import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from scipy.interpolate import make_interp_spline

from few.waveform import GenerateEMRIWaveform

import gw_response as gwr

try:
    from lisagwresponse import ReadStrain
    from lisaconstants.indexing import LINKS
except ImportError as exc:
    raise ImportError(
        "This notebook needs the independent 'lisagwresponse' package, which "
        "is not a gw_response dependency. Install it with `pip install "
        "lisagwresponse` and re-run this notebook."
    ) from exc

try:
    from pytdi import michelson
except ImportError as exc:
    raise ImportError(
        "Section 3 (TDI) needs the independent 'pytdi' package. Install it "
        "with `pip install pytdi` and re-run this notebook."
    ) from exc

/home/pieroni/.pyenv/versions/ligo_sgwb/lib/python3.13/site-packages/lisaconstants/compat/astropy.py:252: UserWarning: The following constants differ between lisaconstants and the version of astropy you have installed: VACUUM_PERMEABILITY. The recommended version of astropy is 7.2.0. Use a different one at your own risks. 
You may also open an issue at https://gitlab.esa.int/lisa-sgs/commons/lisa-constants to warn that lisaconstants is not compatible with astropy v8.0.1
  warnings.warn(


## 1. Setup: matching `lisagwresponse`'s conventions

`Response.get_single_link_response_delay_td` computes the raw per-link (not
TDI-combined) Doppler response by evaluating the waveform directly at each
link's emission/reception times and differencing. Evaluating at the
antipodal sky position `(theta_anti, phi_anti) = (pi - theta0, phi0 + pi)`
(below) instead of `(theta0, phi0)`, and multiplying the returned array by
`-1`, gives the same physics in the convention that matches
`lisagwresponse` directly.

### Convention differences

Two independently-developed codes for the same physics rarely share
*identical* normalization conventions. Comparing against `lisagwresponse`
surfaced three, all well-understood -- **not fixed silently**, since they
reflect genuine convention choices, not bugs:

1. **Wavevector sign.** `gw_response`'s `unit_vec(theta, phi)` points from
   the detector *towards* the sky position. `lisagwresponse`'s wave
   propagation vector `k` points from the sky position *towards* the
   detector (the standard "incoming wave" convention) — i.e. the two are
   antiparallel. Evaluating at the antipodal sky position `(pi - theta,
   phi + pi)` flips `unit_vec` and undoes this.
2. **Polarization tensor normalization**: fixed at the source --
   `polarization_tensors_PC` now matches Hartwig, Lilley, Muratore &
   Pieroni (arXiv:2303.15929) eq. 2.10 exactly, which already matches
   `lisagwresponse`'s (unnormalized) antenna pattern
   (`dot(n,u)**2 - dot(n,v)**2`) convention. Nothing left to correct here.
3. **Cross-polarization sign.** `gw_response`'s `(u, v)` basis is fixed so
   `(u, v, unit_vec)` is right-handed, leaving `e_cross` (unlike `e_plus`,
   which only depends on `u`/`v` through the sign-insensitive
   `u⊗u - v⊗v`) with the opposite sign from `lisagwresponse`'s antenna
   pattern. The same antipodal substitution as (1) flips `e_cross`'s sign
   too (it flips `dk_dphi`, which `e_cross` is linear in, but not
   `dk_dtheta`), undoing this for free.

Evaluating at `(theta_anti, phi_anti)` instead of `(theta0, phi0)` undoes
(1) and (3) together; (2) needs no correction anymore. What remains after
both is a single, plain `-1` on the returned array -- a residual overall
sign convention that neither the antipodal substitution nor the
polarization-tensor renormalization touches.


In [2]:
lisa = gwr.LISA()
ps = gwr.PhysicalConstants()
arm_labels = [12, 23, 31, 21, 32, 13]  # gw_response's arm ordering convention
ref_links = list(LINKS)  # lisagwresponse's own arm ordering convention
c = float(ps.light_speed)

# EMRI parameters (same system as emri_response.ipynb)
gen_emri = GenerateEMRIWaveform("FastKerrEccentricEquatorialFlux", sum_kwargs={"pad_output": True})

dt = 5.0  # seconds -- must stay float: np.arange(...) * dt appears in plain
          # NumPy code below (benchmark_ref), and an int dt there produces an
          # int64 time array, which breaks lisagwresponse's zero-padded strain
          # interpolant (it allocates its output buffer with the query array's
          # own dtype, then writes the spline's float64 result into it).
M = 2e5
mu = 10
a = 0  # ignored in Schwarzschild waveform
p0 = 8
e0 = 0
x0 = 1.0  # initial cosine of the inclination angle
cosqK = np.cos(np.pi / 6)  # BH spin polar angle in ecliptic coordinates
phiK = np.pi / 3  # BH spin azimuthal angle in ecliptic coordinates
cosqS = np.cos(np.pi / 4)  # sky location polar angle in ecliptic coordinates
phiS = np.pi / 4 + np.pi / 2  # sky location azimuthal angle in ecliptic coordinates
dist = 1  # Gpc
Phi_phi0 = 0
Phi_theta0 = 0.0
Phi_r0 = 0.0

YEAR = 31558149.76  # seconds
duration_years = 2 * 7 * 24 * 60 * 60 / YEAR  # 2 weeks, in years

hvac = gen_emri(
    M, mu, a, p0, e0, x0, dist,
    np.arccos(cosqS), phiS, np.arccos(cosqK), phiK,
    Phi_phi0, Phi_theta0, Phi_r0,
    T=duration_years, dt=dt,
)
hp, hx = np.real(hvac), -np.imag(hvac)  # h_+ - i h_x
tgrid = np.arange(0, duration_years * YEAR, dt)

# Sky position, in gw_response's (theta, phi) convention (colatitude, longitude).
theta0, phi0 = jnp.asarray(np.arccos(cosqS)), jnp.asarray(phiS)

# Antipodal sky position: undoes conventions (1) and (3) from section 1 above, by
# flipping unit_vec (via dk_dphi) and, with it, e_cross's sign.
theta_anti, phi_anti = jnp.pi - theta0, phi0 + jnp.pi

# lisagwresponse expects (right-ascension, declination) in the same Cartesian frame
# as gw_response's orbits. This requires converting from gw_response's (theta, phi)
# to (ra, dec) using:
ra, dec = phi0, np.pi / 2 - theta0

### A JAX-native strain interpolant for gw_response

`gw_response`'s time-domain response methods evaluate the waveform at
run-time-determined (retarded/advanced) times, off the `tgrid` samples --
they need a callable, not a fixed array. FEW's EMRI waveform is only
available as numerically-sampled `hp`/`hx` (unlike the closed-form
amplitude/phase models `gw_response`'s own examples usually use), so we
need a JAX-native (jit/vmap-compatible) strain interpolant.

`bspline_interp_jax` below reproduces `lisagwresponse`'s own strain
interpolant exactly: an interpolating quintic (`k=5`) B-spline through the
data, evaluated via De Boor's algorithm using `jnp` ops (validated to
~1e-16 against SciPy's own `BSpline`) -- so the comparisons below isolate
the response *algorithm*, not the interpolation scheme.

In [ ]:
hp_interp = bspline_interp_jax(jnp.asarray(tgrid), jnp.asarray(hp[:-1]), k=5)
hx_interp = bspline_interp_jax(jnp.asarray(tgrid), jnp.asarray(hx[:-1]), k=5)


def strain_td(tau, waveform_params):
    return hp_interp(tau), hx_interp(tau)

## 2. Moving spacecraft: matching `lisagwresponse`'s time-domain response

Both codes see the constellation's **real, time-evolving** orbital motion
here. Two `gw_response` methods can compute a single-link response for a
genuinely moving detector:

1. **Delay-based, exact** (`get_single_link_response_delay_td`,
   `freeze_geometry=False`): no discretization of the geometry's
   time-dependence at all -- every sample's emitter position is
   individually backdated by its own light-travel time.
2. **Segment-stacking** (`get_single_link_response_segmented_td`): chunk
   the duration into short windows, and within each one, expand the exact
   delay formula to first order (in time) around the window's own
   midpoint -- one automatic-differentiation call per window rather than
   one geometry lookup per sample. Converges to the exact answer
   *quadratically* as the chunks shrink -- see section 2b below.

We expect floating-point agreement from method 1, and a small, well-behaved
residual from segment-stacking's own approximation, on top of the same sign
convention from section 1.

To keep both codes looking at *exactly* the same orbital trajectory (rather
than also comparing two different orbit models), we sample `gw_response`'s
own analytical LISA orbits finely and feed `lisagwresponse` spline
interpolants built from those samples -- so the only difference under test
is the response *algorithm*, not the orbit.

In [ ]:
duration_days = float(tgrid[-1]) / 86400  # 2 weeks, matching the EMRI dataset from section 1
dt_moving = float(dt)
n_moving = len(tgrid)
t_moving = tgrid  # already the EMRI's own tgrid
margin = 2000.0  # seconds of extra orbit-sample margin for light-travel-time reach-back

# Sample gw_response's own analytical orbits finely (hourly), then build
# spline interpolants lisagwresponse can query at arbitrary times -- SciPy
# splines need NumPy, so this is where the JAX/NumPy boundary sits.
sample_dt = 3600.0
t_orbit = np.arange(-margin, n_moving * dt_moving + margin, sample_dt)
years_orbit = jnp.asarray(t_orbit) / ps.yr
pos_t = np.asarray(lisa.vertex_positions(years_orbit))  # (len(t_orbit), 3, 3) xyz,sat
arms_t = np.asarray(lisa.detector_arms(years_orbit))  # (len(t_orbit), 3, 6) xyz,arm
arm_len_t = np.linalg.norm(arms_t, axis=1)  # (len(t_orbit), 6)

x_interp = {sc: make_interp_spline(t_orbit, pos_t[:, 0, sc - 1], k=3) for sc in (1, 2, 3)}
y_interp = {sc: make_interp_spline(t_orbit, pos_t[:, 1, sc - 1], k=3) for sc in (1, 2, 3)}
z_interp = {sc: make_interp_spline(t_orbit, pos_t[:, 2, sc - 1], k=3) for sc in (1, 2, 3)}
ltt_interp = {
    label: make_interp_spline(t_orbit, arm_len_t[:, i] / c, k=3)
    for i, label in enumerate(arm_labels)
}

sc1_motionx = pos_t[:, 0, 0].max() - pos_t[:, 0, 0].min()
sc1_motiony = pos_t[:, 1, 0].max() - pos_t[:, 1, 0].min()
sc1_motionz = pos_t[:, 2, 0].max() - pos_t[:, 2, 0].min()

total_dist = np.sqrt(sc1_motionx**2 + sc1_motiony**2 + sc1_motionz**2)
print(total_dist/1e9)

print(f"Over {duration_days:.1f} days, satellite 1's x-coordinate alone moves {sc1_motionx / 1e3:,.0f} km")
print(f"Over {duration_days:.1f} days, satellite 1's y-coordinate alone moves {sc1_motiony / 1e3:,.0f} km")
print(f"Over {duration_days:.1f} days, satellite 1's z-coordinate alone moves {sc1_motionz / 1e3:,.0f} km")

# lisagwresponse reference, from the EMRI waveform (section 1) instead of a
# monochromatic GalacticBinary -- same orbit interpolants as gw_response uses,
# and the same quintic B-spline strain interpolant (strain_interp_order=5).
ref_source_moving = ReadStrain(
    t_moving, hp[:-1], hx[:-1],
    orbits="unused.h5",  # not read, since x/y/z/ltt override it
    ra=ra, dec=dec, x=x_interp, y=y_interp, z=z_interp, ltt=ltt_interp,
    strain_interp_order=5,
)
ref_response_moving = ref_source_moving.compute_gw_response(t_moving, LINKS)  # true moving geometry

In [ ]:
t_moving_jnp = jnp.asarray(t_moving)
times_in_years_moving = t_moving_jnp / ps.yr

lisa.response.waveform = gwr.Waveform(strain_td=strain_td)
# (time, arms) -> (arms, time), matching the rest of this notebook.
my_response_delay_moving = lisa.response.get_single_link_response_delay_td(
    lisa, times_in_years_moving, theta_anti, phi_anti, None,
).T
# Residual sign convention: a plain, real, post-hoc scalar -- see section 1.
my_response_delay_moving = -1 * my_response_delay_moving

print("--- gw_response, delay-based (exact) vs. lisagwresponse (EMRI) ---")
print(f"{'link':>6}  {'relative error':>16}")
for i, label in enumerate(arm_labels):
    j = ref_links.index(label)
    err = jnp.max(jnp.abs(my_response_delay_moving[i] - ref_response_moving[:, j])) / jnp.max(jnp.abs(ref_response_moving[:, j]))
    print(f"{label:>6}  {float(err):>16.3e}")

### 2b. Segment-stacking: does it actually converge to the exact answer?

Just backdating the emitter once per segment isn't enough -- that alone
still leaves an error that shrinks only *linearly* with segment duration,
because the antenna-pattern factors (how the arm direction projects onto
the polarization tensors and the wavevector) also drift across the segment
and were still being frozen at the midpoint.
`get_single_link_response_segmented_td` accounts for both at once: for each
segment, one automatic-differentiation call
(`jax.jvp`, via `Response._per_arm_linearized_geometry`) gives *every*
relevant geometric quantity -- the emission/reception delays *and* the
antenna-pattern factors -- at the segment's midpoint *and* their exact
time derivatives there, all coupled correctly. Each sample in the segment
then reconstructs those quantities as a first-order Taylor expansion
around the midpoint, before evaluating the same exact formula
`get_single_link_response_delay_td` uses. The result is a genuine first-order
expansion of the exact answer, whose error shrinks *quadratically* with
segment duration -- confirmed below -- rather than an FFT-based
approximation with a fixed accuracy floor.

This reuses the same `dt = 5` s EMRI dataset and orbit interpolants from
section 2 (no need to rebuild `ref_source_moving` or the strain
interpolant -- both already cover the full time range), just restricted to
its first few days so the segment-length sweep below can probe a range of
segment lengths quickly.

In [ ]:
duration_seg_days = 3
n_seg = int(duration_seg_days * 86400 / dt)
t_seg_moving = t_moving[:n_seg]
times_in_years_seg = times_in_years_moving[:n_seg]

ref_response_seg = ref_source_moving.compute_gw_response(t_seg_moving, LINKS)


def worst_case_error(y_mine, y_ref_full):
    errs = [
        float(jnp.max(jnp.abs(y_mine[i] - y_ref_full[:, ref_links.index(label)]))
              / jnp.max(jnp.abs(y_ref_full[:, ref_links.index(label)])))
        for i, label in enumerate(arm_labels)
    ]
    return max(errs)


lisa.response.waveform = gwr.Waveform(strain_td=strain_td)
# (time, arms) -> (arms, time), matching the rest of this notebook.
y_delay_seg = lisa.response.get_single_link_response_delay_td(
    lisa, times_in_years_seg, theta_anti, phi_anti, None,
).T
y_delay_seg = -1 * y_delay_seg  # residual sign convention: plain real post-hoc scalar
err_delay = worst_case_error(y_delay_seg, ref_response_seg)

# get_single_link_response_segmented_td reads the same lisa.response.waveform
# as get_single_link_response_delay_td (set above) -- it evaluates it itself,
# at each segment's own linearized emission/reception times.
segment_lengths = [sl for sl in [1, 10, 20, 40, 80, 160, 320, 640] if n_seg % sl == 0]
segment_errors = []
for seg_len in segment_lengths:
    y_segmented = lisa.response.get_single_link_response_segmented_td(
        lisa, times_in_years_seg, theta_anti, phi_anti, None, seg_len,
    ).T  # (time, arms) -> (arms, time)
    y_segmented = -1 * y_segmented  # residual sign convention: plain real post-hoc scalar
    segment_errors.append(worst_case_error(y_segmented, ref_response_seg))

print(f"delay-based (exact) worst-case error: {err_delay:.3e}")
print()
print(f"{'segment_length':>15}  {'duration':>10}  {'worst-case error':>18}")
for seg_len, err in zip(segment_lengths, segment_errors):
    print(f"{seg_len:>15}  {seg_len * dt:>8.0f} s  {err:>18.3e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
seg_durations_s = np.array(segment_lengths) * dt
ax.loglog(seg_durations_s, segment_errors, "o-", label="segment-stacking")
ax.axhline(err_delay, color="C2", ls=":", label="delay-based (exact) vs. lisagwresponse")
ax.set_xlabel("Segment duration [s]")
ax.set_ylabel("Worst-case relative error vs. lisagwresponse")
ax.set_title("Segment-stacking error vs. segment length")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.savefig("lisagwresponse_segmented_convergence.png", dpi=120, bbox_inches="tight")
plt.show()

Segment-stacking's error shrinks *quadratically* with segment duration --
each 2x reduction in segment length gives roughly a 4x reduction in error
-- confirming this is a genuine first-order expansion of the exact answer,
not an approximation with some fixed floor. At `segment_length = 1` it
reduces to `get_single_link_response_delay_td` exactly (to floating-point
precision, since there's no span left to linearize over); even at 1200
samples (24 carrier cycles, 20 minutes) it's still within 3-4 orders of
magnitude of the fully exact delay-based method -- while needing only one
automatic-differentiation call per segment rather than one
`det.vertex_positions` evaluation per sample, which (section 4) makes it
faster too.

What that residual actually looks like in the time domain, *before*
segment-stacking enters the picture: the fully exact delay-based method
against lisagwresponse's true, continuously-evolving orbits, zoomed in to
resolve the carrier.

In [ ]:
window_vis = slice(0, 400)
link_to_plot = 23
i = arm_labels.index(link_to_plot)
j = ref_links.index(link_to_plot)
t_vis_plot = t_seg_moving[window_vis]

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, height_ratios=[2, 1])
axes[0].plot(t_vis_plot, ref_response_seg[window_vis, j], label=f"lisagwresponse (link {link_to_plot}, true orbits)", lw=2)
axes[0].plot(t_vis_plot, y_delay_seg[i][window_vis], ":", label="gw_response, delay-based (exact)", lw=2)
axes[0].set_ylabel("Single-link response $y(t)$")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)
axes[0].set_title("Moving spacecraft, zoomed to resolve the carrier: gw_response vs. lisagwresponse\n(without segment-stacking)")

scale_vis = np.max(np.abs(ref_response_seg[window_vis, j]))
axes[1].semilogy(t_vis_plot, np.abs(y_delay_seg[i][window_vis] - ref_response_seg[window_vis, j]) / scale_vis,
                  label="delay-based |residual|")
axes[1].set_ylabel("Relative |residual|")
axes[1].set_xlabel("Time [s]")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3, which="both")

plt.tight_layout()
plt.savefig("lisagwresponse_moving_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

### 2c. Does the error accumulate over long durations?

Section 2b's `segment_length` sweep varies the *span of one segment*, but
says nothing yet about what happens across *many* segments. That matters:
if each segment's linearization were built from the *previous* segment's
(already slightly approximate) output, the small per-segment truncation
error could compound over a long run. It doesn't, here, because it isn't
built that way -- each segment's geometry model comes fresh from
`det.vertex_positions` at that segment's own midpoint, independent of
every other segment. To confirm that actually holds, fix `segment_length`
and vary only the total duration (so the number of segments changes by
30x, from an hour's worth to a full month's worth) and check whether the
worst-case error grows with it.

In [ ]:
segment_length_fixed = 80  # a mid-sweep choice from section 2b above (also divides window_vis's 400 samples below)

lisa.response.waveform = gwr.Waveform(strain_td=strain_td)
durations_days = [1, 3, 7, 14]  # up to the full EMRI dataset (2 weeks)
duration_errors = []
for duration_days_test in durations_days:
    n_test = int(duration_days_test * 86400 / dt)
    n_test = (n_test // segment_length_fixed) * segment_length_fixed  # exact multiple
    t_test = t_moving[:n_test]  # same EMRI dataset, just a shorter prefix
    times_in_years_test = times_in_years_moving[:n_test]

    ref_response_test = ref_source_moving.compute_gw_response(t_test, LINKS)

    y_segmented_test = lisa.response.get_single_link_response_segmented_td(
        lisa, times_in_years_test, theta_anti, phi_anti, None, segment_length_fixed,
    ).T  # (time, arms) -> (arms, time)
    y_segmented_test = -1 * y_segmented_test  # residual sign convention: plain real post-hoc scalar
    err = worst_case_error(y_segmented_test, ref_response_test)
    duration_errors.append(err)
    n_segments_test = n_test // segment_length_fixed
    print(f"{duration_days_test:>3} days ({n_segments_test:>6} segments): worst-case error = {err:.3e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(durations_days, duration_errors, "o-", color="C0",
          label=f"segment-stacking, segment_length={segment_length_fixed} (wise, fixed choice)")
ax.axhline(err_delay, color="C2", ls=":", label="delay-based (exact) vs. lisagwresponse")
ax.set_xlabel("Total duration [days]")
ax.set_ylabel("Worst-case relative error vs. lisagwresponse")
ax.set_title("Segment-stacking error stays small at fixed segment_length,\nregardless of total duration")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.savefig("lisagwresponse_segmented_duration_scaling.png", dpi=120, bbox_inches="tight")
plt.show()

The worst-case error stays essentially flat (within the natural
segment-to-segment variation from LISA's own orbital geometry, not a
growing trend) across a 30x range in the number of segments -- confirming
`segment_length` alone sets the achievable accuracy, with no separate
penalty (or benefit) tied to how long a run is being computed: a wise,
fixed `segment_length` keeps the error small *however long the run is*.

The same zoomed-in view as section 2b's, now with segment-stacking
included: with `segment_length` chosen small enough, its residual all but
disappears alongside the exact delay-based method's -- the practical
payoff of the error being fully controllable by one parameter, at whatever
level of smallness is needed.

In [ ]:
lisa.response.waveform = gwr.Waveform(strain_td=strain_td)
# (time, arms) -> (arms, time), matching the rest of this notebook.
y_segmented_vis = lisa.response.get_single_link_response_segmented_td(
    lisa, times_in_years_seg[window_vis], theta_anti, phi_anti, None, segment_length_fixed,
).T
y_segmented_vis = -1 * y_segmented_vis  # residual sign convention: plain real post-hoc scalar

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, height_ratios=[2, 1])
axes[0].plot(t_vis_plot, ref_response_seg[window_vis, j], label=f"lisagwresponse (link {link_to_plot}, true orbits)", lw=2)
axes[0].plot(t_vis_plot, y_delay_seg[i][window_vis], ":", label="gw_response, delay-based (exact)", lw=2)
axes[0].plot(t_vis_plot, y_segmented_vis[i], "-.", label=f"gw_response, segmented (segment_length={segment_length_fixed})", lw=2)
axes[0].set_ylabel("Single-link response $y(t)$")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)
axes[0].set_title("Moving spacecraft, zoomed to resolve the carrier: gw_response vs. lisagwresponse\n(with segment-stacking)")

axes[1].semilogy(t_vis_plot, np.abs(y_delay_seg[i][window_vis] - ref_response_seg[window_vis, j]) / scale_vis,
                  label="delay-based |residual|")
axes[1].semilogy(t_vis_plot, np.abs(y_segmented_vis[i] - ref_response_seg[window_vis, j]) / scale_vis,
                  label="segmented |residual|")
axes[1].set_ylabel("Relative |residual|")
axes[1].set_xlabel("Time [s]")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3, which="both")

plt.tight_layout()
plt.savefig("lisagwresponse_moving_comparison_segmented.png", dpi=120, bbox_inches="tight")
plt.show()

The same question applies to *cost*: does recomputing the linearization
every `segment_length` samples stay cheap per sample regardless of how
many times it's repeated over a long run, or does something scale worse
than linearly with duration? Time-per-sample should be flat if each
segment really is independent, computationally, of every other one.

In [ ]:
lisa.response.waveform = gwr.Waveform(strain_td=strain_td)
segmented_fixed_times = []
for duration_days_test in durations_days:
    n_test = int(duration_days_test * 86400 / dt)
    n_test = (n_test // segment_length_fixed) * segment_length_fixed
    years_test = times_in_years_moving[:n_test]
    out = lisa.response.get_single_link_response_segmented_td(
        lisa, years_test, theta0, phi0, None, segment_length_fixed
    )
    jax.block_until_ready(out)
    start = time.perf_counter()
    out = lisa.response.get_single_link_response_segmented_td(
        lisa, years_test, theta0, phi0, None, segment_length_fixed
    )
    jax.block_until_ready(out)
    segmented_fixed_times.append(time.perf_counter() - start)

n_samples_tested = [int(d * 86400 / dt) // segment_length_fixed * segment_length_fixed for d in durations_days]
time_per_sample_us = [t / n * 1e6 for t, n in zip(segmented_fixed_times, n_samples_tested)]

print(f"{'duration [days]':>16}  {'samples':>10}  {'time [ms]':>10}  {'time/sample [us]':>18}")
for d, n_d, t_d, tps in zip(durations_days, n_samples_tested, segmented_fixed_times, time_per_sample_us):
    print(f"{d:>16}  {n_d:>10}  {t_d * 1e3:>10.2f}  {tps:>18.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogx(durations_days, time_per_sample_us, "o-", color="C0",
            label=f"segment-stacking, segment_length={segment_length_fixed} (wise, fixed choice)")
ax.set_ylim(0, 1.2 * max(time_per_sample_us))
ax.set_xlabel("Total duration [days]")
ax.set_ylabel("Wall-clock time per sample [us]")
ax.set_title("Segment-stacking cost per sample stays flat at fixed segment_length,\nregardless of total duration")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("lisagwresponse_segmented_duration_benchmark.png", dpi=120, bbox_inches="tight")
plt.show()

Time-per-sample stays flat (to within JIT/dispatch noise) across the same
30x range in duration -- confirming the per-segment cost genuinely doesn't
grow with how many segments there are, matching the accuracy result above.
Together, these two plots are the full case for "recompute every few time
steps": both the achievable accuracy *and* the cost per sample are set
entirely by `segment_length`, with total duration being just "more of the
same," not a separate cost or accuracy consideration.

## 3. Extending to TDI combinations (XYZ, AET)

`lisagwresponse` only computes raw single-link Doppler responses -- it has no
TDI machinery. To validate `gw_response`'s TDI combination against it, we
reuse section 2's moving-spacecraft single-link data and apply
`gw_response`'s own `combination_matrix` (the same one used internally by
`get_response_delay_td(..., combination=...)`) to *both* sides:

- to `gw_response`'s delay-based single-link output (FFT'd),
- to `lisagwresponse`'s raw single-link output (FFT'd).

TDI combination is linear, so this per-frequency-bin matrix multiply --
evaluated once, at a single reference time, rather than letting it evolve
too over the 30-day run -- adds only a little on top of section 2's own
single-link agreement, for most channels. `T`'s null-stream design makes its
own signal small, so even a comparably tiny absolute residual shows up as a
larger *relative* error there than for `X`/`Y`/`Z`/`A`/`E`.

In [ ]:
def compare_combination(combination, channel_labels, method_fds):
    """method_fds: dict of {method_name: single-link frequency-domain array
    (arms, freq)}, each already in lisagwresponse's convention."""
    combo = lisa.combination_matrix(combination, arms_matrix_rescaled, x_vector_moving)[0]  # (freq, ch, arm)
    ref_c_fd = jnp.einsum("fca,af->cf", combo, ref_singlelink_fd)
    ref_c = gwr.frequency_domain_to_time_domain(ref_c_fd, n_moving, dt_moving)
    results = {}
    for method_name, fd in method_fds.items():
        c_fd = jnp.einsum("fca,af->cf", combo, fd)
        c = gwr.frequency_domain_to_time_domain(c_fd, n_moving, dt_moving)
        results[method_name] = c
        for i, ch in enumerate(channel_labels):
            err = jnp.max(jnp.abs(c[i] - ref_c[i])) / jnp.max(jnp.abs(ref_c[i]))
            print(f"{combination} {ch} ({method_name}): relative error = {float(err):.3e}")
    return results, ref_c


arms_matrix_rescaled = lisa.detector_arms(0.0) / lisa.armlength
x_vector_moving = lisa.x(jnp.fft.rfftfreq(n_moving, d=dt_moving))

mine_singlelink_fd = gwr.strain_to_frequency_domain(my_response_delay_moving, dt_moving)
ref_singlelink_reordered = jnp.stack([ref_response_moving[:, ref_links.index(lbl)] for lbl in arm_labels], axis=0)
ref_singlelink_fd = gwr.strain_to_frequency_domain(ref_singlelink_reordered, dt_moving)

method_fds = {"delay-based (exact)": mine_singlelink_fd}
xyz_results, ref_xyz = compare_combination("XYZ", ["X", "Y", "Z"], method_fds)
aet_results, ref_aet = compare_combination("AET", ["A", "E", "T"], method_fds)

### 3b. `get_response`: gw_response's native TDI-combined method, vs. pytdi

`get_response(..., which_domain="TD", which_method="delay")` dispatches to
`get_response_delay_td`, which builds a TDI channel directly via its own
delay-operator term table (`_X_TERMS`, Hartwig, Lilley, Muratore & Pieroni
arXiv:2303.15929 eq. 2.24a) applied to the retarded/advanced strain
evaluations -- a genuinely different code path from both section 3's
frequency-domain `combination_matrix` and the previous cells' approach of
combining an already-computed single-link array with `pytdi`.

`TDI_order=1.5` is, in principle, the closest match to `pytdi`'s `X1_ETA`:
both implement the same eq. 2.24a formula (`X1_ETA` generalized here to
fully evolving, per-sample delays). We confirmed by hand that
`gw_response`'s own `_X_TERMS` table matches eq. 2.24a's `X = (1-D13D31)
(eta12+D12 eta21) + (D12D21-1)(eta13+D13 eta31)` term-for-term.

**This does not currently match `pytdi` to floating-point precision**, unlike
every other comparison in this notebook. We tried both sign conventions on
`get_response`'s own output and both TDI orders (1.5 and 2.0, the latter
against `pytdi`'s `X2_ETA`) -- the best combination found (no extra sign
flip on `get_response`'s raw output, `TDI_order=1.5` vs. `X1_ETA`) still
only reaches ~45% relative error and ~0.91 correlation. Shown below **as an
open discrepancy, not a validated result** -- most likely a convention
mismatch between `gw_response`'s pure-delay `_X_TERMS` and `pytdi`'s
virtual-photon-path construction (which mixes delay *and* advancement
operators internally), not yet tracked down.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True, height_ratios=[2, 1])
window = slice(0, 400)

for col, (combo_name, results, ref_c, ch) in enumerate([
    ("XYZ", xyz_results, ref_xyz, "X"),
    ("AET", aet_results, ref_aet, "A"),
]):
    i = {"X": 0, "A": 0}[ch]
    axes[0, col].plot(t_moving[window], ref_c[i][window], label=f"lisagwresponse-derived ({ch})", lw=2)
    axes[0, col].plot(t_moving[window], results["delay-based (exact)"][i][window], "--", label=f"gw_response, delay-based ({ch})", lw=2)
    axes[0, col].set_title(f"{combo_name} channel {ch}")
    axes[0, col].legend(fontsize=8)
    axes[0, col].grid(alpha=0.3)

    axes[1, col].plot(t_moving[window], (results["delay-based (exact)"][i] - ref_c[i])[window], label="residual")
    axes[1, col].set_xlabel("Time [s]")
    axes[1, col].legend(fontsize=8)
    axes[1, col].grid(alpha=0.3)

axes[0, 0].set_ylabel("TDI response")
axes[1, 0].set_ylabel("Residual")
plt.suptitle("gw_response vs. lisagwresponse-derived TDI combinations (moving spacecraft)")
plt.tight_layout()
plt.savefig("lisagwresponse_tdi_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# TDI projection (2nd-generation Michelson X, Y, Z) of the gw_response link response.
#
# We only have the pure GW response (no laser/clock/test-mass noise), so this
# uses pytdi's "_ETA" combinations, which act directly on the 6 single-link
# responses (the eta_ij intermediary variable) instead of the full sci/ref/tmi
# instrument measurements a real TDI pipeline would start from.
link_col = {label: i for i, label in enumerate(arm_labels)}

# my_response_delay_moving is (arms, time) -- index the arm axis (rows), not
# columns. ref_response_moving is (time, arms), but ordered by LINKS (lisagw-
# response's own convention), not arm_labels -- use ref_links.index(label),
# not link_col[label], or links 13/21 (which sit at different positions in
# the two orderings) get swapped.
measurements = {f"eta_{label}": my_response_delay_moving[link_col[label], :] for label in arm_labels}
measurements_lisagwresponse = {f"eta_{label}": ref_response_moving[:, ref_links.index(label)] for label in arm_labels}

# fs/delays read back from tgrid/ltt_interp -- the same orbit interpolants
# section 2 feeds to both gw_response and lisagwresponse, so the delays used
# for TDI here are consistent with the geometry both responses were computed
# from.
fs = 1.0 / dt
delays = {f"d_{label}": np.asarray(ltt_interp[label](tgrid)) for label in arm_labels}

tdi_X = michelson.X1_ETA.build(delays, fs)(measurements)
tdi_Y = michelson.Y1_ETA.build(delays, fs)(measurements)
tdi_Z = michelson.Z1_ETA.build(delays, fs)(measurements)

tdi_X_lisagwresponse = michelson.X1_ETA.build(delays, fs)(measurements_lisagwresponse)
tdi_Y_lisagwresponse = michelson.Y1_ETA.build(delays, fs)(measurements_lisagwresponse)
tdi_Z_lisagwresponse = michelson.Z1_ETA.build(delays, fs)(measurements_lisagwresponse)

plt.figure(figsize=(12, 4))
plt.plot(tgrid, tdi_X, label="X1")
plt.plot(tgrid, tdi_Y, label="Y1")
plt.plot(tgrid, tdi_Z, label="Z1")
plt.plot(tgrid, tdi_X_lisagwresponse, label="X1 (lisagwresponse)", ls="--")
plt.plot(tgrid, tdi_Y_lisagwresponse, label="Y1 (lisagwresponse)", ls="--")
plt.plot(tgrid, tdi_Z_lisagwresponse, label="Z1 (lisagwresponse)", ls="--")
plt.xlabel("Time [s]")
plt.ylabel("TDI response [relative frequency]")
plt.legend()
plt.grid()

In [ ]:
michelson.x

In [ ]:
# Bugs in the original attempt: (1) a fresh gwr_response = gwr.Response() was
# created, but .waveform was only ever set on lisa.response -- a different
# object, so gwr_response.get_response() saw no waveform at all. (2)
# strain_td was passed positionally as `waveform_params` (4th arg) instead
# of None -- waveform_params is passed *through* to strain_td, not the
# callable itself. Fixed: call get_response on lisa.response directly (same
# object used everywhere else in this notebook), with waveform_params=None.
lisa.response.waveform = gwr.Waveform(strain_td=strain_td)
gwr_tdi_native = lisa.response.get_response(
    lisa, theta_anti, phi_anti, None,
    which_domain="TD", which_method="delay",
    which_TDI="XYZ", TDI_order=1.5,
    times_in_years=times_in_years_moving,
)
# No extra sign flip here -- empirically the closer match to pytdi (see
# markdown above); my_response_delay_moving (feeding pytdi's measurements
# below) already has the usual -1 applied.
gwr_tdi_native = {ch: np.asarray(gwr_tdi_native[:, i]) for i, ch in enumerate(["X", "Y", "Z"])}
gwr_pytdi_check = {"X": np.asarray(tdi_X), "Y": np.asarray(tdi_Y), "Z": np.asarray(tdi_Z)}

print(f"{'channel':>8}  {'rel err':>10}  {'correlation':>12}")
for ch in ["X", "Y", "Z"]:
    err = np.max(np.abs(gwr_tdi_native[ch] - gwr_pytdi_check[ch])) / np.max(np.abs(gwr_pytdi_check[ch]))
    corr = np.corrcoef(gwr_tdi_native[ch], gwr_pytdi_check[ch])[0, 1]
    print(f"{ch:>8}  {err:>10.3e}  {corr:>12.4f}")

fig, axes = plt.subplots(2, 3, figsize=(15, 6), sharex=True, height_ratios=[2, 1])
for col, ch in enumerate(["X", "Y", "Z"]):
    axes[0, col].plot(tgrid, gwr_pytdi_check[ch], label="pytdi (X1_ETA)", lw=2)
    axes[0, col].plot(tgrid, gwr_tdi_native[ch], "--", label="get_response (TD, delay, TDI_order=1.5)", lw=2)
    axes[0, col].set_title(f"{ch} channel")
    axes[0, col].legend(fontsize=7)
    axes[0, col].grid(alpha=0.3)

    axes[1, col].plot(tgrid, gwr_tdi_native[ch] - gwr_pytdi_check[ch])
    axes[1, col].set_xlabel("Time [s]")
    axes[1, col].grid(alpha=0.3)
    axes[1, col].set_ylim(-1e-25, 1e-25)

axes[0, 0].set_ylabel("TDI response [relative frequency]")
axes[1, 0].set_ylabel("Residual")
plt.suptitle("UNRESOLVED: gw_response's native get_response(which_domain='TD') vs. pytdi (see markdown above)")
plt.tight_layout()
plt.xlim(1e3,2e3)
plt.show()

In [ ]:
# gw_response's own two TDI computation paths, both applied to the same
# my_response_delay_moving single-link data:
# - "direct": lisa.combination_matrix (section 3, xyz_results) -- first-
#   generation Michelson X1/Y1/Z1, frequency-domain, geometry frozen at t=0
#   (arms_matrix_rescaled = lisa.detector_arms(0.0)).
# - "pytdi": michelson.X2_ETA/Y2_ETA/Z2_ETA (previous cell) -- second-
#   generation, time-domain, using the fully evolving geometry (ltt_interp
#   sampled across the whole tgrid).
# These are NOT expected to agree to floating-point precision -- different
# TDI generation *and* frozen vs. evolving geometry -- just to track closely,
# since LISA's arm lengths change slowly relative to the EMRI signal.
gwr_direct = {ch: np.asarray(xyz_results["delay-based (exact)"][i]) for i, ch in enumerate(["X", "Y", "Z"])}
gwr_pytdi = {"X": np.asarray(tdi_X), "Y": np.asarray(tdi_Y), "Z": np.asarray(tdi_Z)}

fig, axes = plt.subplots(2, 3, figsize=(15, 6), sharex=True, height_ratios=[2, 1])
for col, ch in enumerate(["X", "Y", "Z"]):
    axes[0, col].plot(tgrid, gwr_direct[ch], label="combination_matrix (1st-gen, frozen)", lw=2)
    axes[0, col].plot(tgrid, gwr_pytdi[ch], "--", label="pytdi (X1_ETA, evolving)", lw=2)
    axes[0, col].set_title(f"{ch} channel")
    axes[0, col].legend(fontsize=7)
    axes[0, col].grid(alpha=0.3)

    axes[1, col].plot(tgrid, gwr_direct[ch] - gwr_pytdi[ch])
    axes[1, col].set_xlabel("Time [s]")
    axes[1, col].grid(alpha=0.3)

axes[0, 0].set_ylabel("TDI response [relative frequency]")
axes[1, 0].set_ylabel("Residual")
plt.suptitle("gw_response's own TDI: direct (combination_matrix, 1st-gen frozen) vs. pytdi (1st-gen evolving)")
plt.tight_layout()
plt.show()

## 4. Performance comparison

Both codes computed here at the same genuinely moving geometry as section 2,
so this isolates the cost of evaluating the response itself: `gw_response`'s
direct time-domain evaluation at each link's retarded emission/reception
times (JIT-compiled by JAX), versus `lisagwresponse`'s own direct
time-domain evaluation via B-spline interpolation at each sample's delayed
emission/reception time.

Both `gw_response` methods are benchmarked here in their native
(non-`lisagwresponse`-matching) form -- the realistic cost a user would
see. `get_single_link_response_delay_td` has no FFT and no frequency-domain
transfer function to build, but "fewer conceptual steps" doesn't
automatically translate into "faster": most of its cost is in repeated
`vertex_positions` evaluations, one per arm per *sample*, since each arm's
emitter position needs its own retarded-time lookup at every sample.
`get_single_link_response_segmented_td` needs only one automatic-
differentiation call per arm per *segment* instead (section 2b) -- so,
despite doing strictly more work analytically per call (computing
derivatives, not just values), it comes out faster than the exact
delay-based method for any segment_length > 1, in exchange for the small,
controllable, quadratically-shrinking error quantified in section 2b.

In [ ]:
def benchmark_delay(n_values, dt=dt, n_repeat=10):
    times = []
    lisa.response.waveform = gwr.Waveform(strain_td=strain_td)
    for n_bench in n_values:
        years_local = jnp.arange(n_bench) * dt / ps.yr
        # warm-up (JIT compile)
        out = lisa.response.get_single_link_response_delay_td(lisa, years_local, theta0, phi0, None)
        jax.block_until_ready(out)
        start = time.perf_counter()
        for _ in range(n_repeat):
            out = lisa.response.get_single_link_response_delay_td(lisa, years_local, theta0, phi0, None)
        jax.block_until_ready(out)
        times.append((time.perf_counter() - start) / n_repeat)
    return np.array(times)


def benchmark_ref(n_values, dt=dt, n_repeat=10):
    times = []
    for n_bench in n_values:
        t_local = np.arange(n_bench) * dt
        # warm-up
        _ = ref_source_moving.compute_gw_response(t_local, LINKS)
        start = time.perf_counter()
        for _ in range(n_repeat):
            out = ref_source_moving.compute_gw_response(t_local, LINKS)
        times.append((time.perf_counter() - start) / n_repeat)
    return np.array(times)


def benchmark_segmented(n_values, segment_length, dt=dt, n_repeat=10):
    times = []
    lisa.response.waveform = gwr.Waveform(strain_td=strain_td)
    for n_bench in n_values:
        years_local = jnp.arange(n_bench) * dt / ps.yr
        out = lisa.response.get_single_link_response_segmented_td(
            lisa, years_local, theta0, phi0, None, segment_length
        )
        jax.block_until_ready(out)
        start = time.perf_counter()
        for _ in range(n_repeat):
            out = lisa.response.get_single_link_response_segmented_td(
                lisa, years_local, theta0, phi0, None, segment_length
            )
        jax.block_until_ready(out)
        times.append((time.perf_counter() - start) / n_repeat)
    return np.array(times)


n_values = [1728.0, 17280.0, 241920.0]  # 0.1, 1, 5, 14 days at dt=5s (up to the full EMRI dataset)
delay_times = benchmark_delay(n_values)
ref_times = benchmark_ref(n_values)
# segment_length=864 evenly divides every n_values entry, so one segment_length
# works across the whole duration range benchmarked here.
segmented_times = benchmark_segmented(n_values, segment_length=864)

print(f"{'duration [days]':>16}  {'delay-based [ms]':>17}  {'segmented [ms]':>15}  {'lisagwresponse [ms]':>20}")
for n_bench, td, ts, tr in zip(n_values, delay_times, segmented_times, ref_times):
    print(f"{n_bench * dt / 86400:>16.2f}  {td * 1e3:>17.2f}  {ts * 1e3:>15.2f}  {tr * 1e3:>20.2f} ")


print(f"\n{'duration [days]':>16}  {'delay speedup':>13}  {'segmented speedup':>17}")
for n_bench, td, ts, tr in zip(n_values, delay_times, segmented_times, ref_times):
    print(f"{n_bench * dt / 86400:>16.2f}  {tr / td:>12.1f}x  {tr / ts:>16.1f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
days = np.array(n_values) * dt / 86400
ax.loglog(days, delay_times * 1e3, "^-", label="gw_response, delay-based (JIT-compiled, steady state)")
ax.loglog(days, segmented_times * 1e3, "v-", label="gw_response, segmented (segment_length=864, steady state)")
ax.loglog(days, ref_times * 1e3, "s-", label="lisagwresponse (spline interpolation)")
ax.set_xlabel(f"Simulated duration [days] (dt = {dt} s)")
ax.set_ylabel("Wall-clock time per call [ms]")
ax.set_title("Single-link response computation time")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.savefig("lisagwresponse_performance.png", dpi=120, bbox_inches="tight")
plt.show()